# INSURED_VALUE != 0

In [ ]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from insurance_data_processor import InsuranceDataProcessor

idp = InsuranceDataProcessor('configs/filters')

df, _ = idp.handle_nans(pd.read_csv('/Users/ilia.stan/MLOps/project/generator/vehicle-insurance-data/motor_data11-14lats.csv'), True)
df_zero, df_else, _ = idp.filter_outliers(df)

df = df_else

categorical_cols = ['USAGE', 'TYPE_VEHICLE', 'MAKE', 'INSR_TYPE']

df_encoded = df[categorical_cols].copy()
df_encoded['INSR_TYPE'] = df_encoded['INSR_TYPE'].apply(lambda x: str(x))

df_binary = pd.get_dummies(df_encoded)

frequent_itemsets = apriori(df_binary, min_support=0.01, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

print(f"Found {len(rules)} rules")

Found 382 rules


В результате просмотра этих правил были выбраны следующие:

Rule 1:
  IF MAKE_YAMAHA
  THEN TYPE_VEHICLE_Motor-cycle
  Confidence: 100.00% | Lift: 14.35

Rule 2:
  IF MAKE_CALABRESE
  THEN TYPE_VEHICLE_Trailers and semitrailers
  Confidence: 100.00% | Lift: 14.31

Rule 3:
  IF MAKE_FORD, USAGE_Own Goods
  THEN TYPE_VEHICLE_Pick-up
  Confidence: 100.00% | Lift: 4.21

Rule 4:
  IF MAKE_BAJAJ
  THEN TYPE_VEHICLE_Motor-cycle
  Confidence: 99.74% | Lift: 14.31

Rule 5:
  IF USAGE_Fare Paying Passengers
  THEN INSR_TYPE_1202
  Confidence: 99.8631 | Lift: 1.516027

Rule 6:
  IF USAGE_General Cartage
  THEN INSR_TYPE_1202
  Confidence: 99.7392 | Lift: 1.514146

# Выводы

правила со 100% уверенностью (например: 1, 2) будут использоваться для проверки качества тестовых данных.

правила 4, 5, 6 будут использоваться для очистки обучающей выборки и проверки качества тестов.

правило 3 вызывает сомнения (FORD производит не только пикапы).

In [2]:
df[df['MAKE'] == 'FORD']

,SEX,INSR_TYPE,INSURED_VALUE,PREMIUM,OBJECT_ID,PROD_YEAR,SEATS_NUM,TYPE_VEHICLE,CCM_TON,MAKE,USAGE,START_MNTH,DURATION,OBJECT_AGE
22,0,1202,131737.0,77.550,5000017912,2002.0,4.0,Pick-up,2892.0,FORD,Own Goods,138,1,114.0
23,0,1202,131737.0,2193.569,5000017912,2002.0,4.0,Pick-up,2892.0,FORD,Own Goods,139,12,115.0
24,0,1202,131737.0,2364.770,5000017912,2002.0,4.0,Pick-up,2892.0,FORD,Own Goods,151,12,127.0
25,0,1202,131737.0,2364.770,5000017912,2002.0,4.0,Pick-up,2892.0,FORD,Own Goods,163,12,139.0
185,0,1202,131737.0,77.550,5000018023,2002.0,4.0,Pick-up,2892.0,FORD,Own Goods,138,1,114.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287560,0,1202,1200000.0,5376.050,5000629405,2011.0,4.0,Pick-up,2499.0,FORD,Own Goods,167,12,35.0
287582,1,1202,1950000.0,14553.390,5000629509,2013.0,4.0,Pick-up,2200.0,FORD,Own Goods,173,12,17.0
287659,0,1202,1600000.0,11626.500,5000629908,2013.0,4.0,Pick-up,2200.0,FORD,Own Goods,173,12,17.0
288279,1,1201,230000.0,4768.180,5000633449,1991.0,4.0,Automobile,1295.0,FORD,Private,173,12,281.0


# INSURED_VALUE == 0

In [3]:
def parse_rule_items(items, prefix_sep):
        result = {}
        for item in items:
            col, val = item.split(prefix_sep)
            result[col] = int(val) if col == 'INSR_TYPE' else val
        return result

In [4]:
df = df_zero

categorical_cols = ['USAGE', 'TYPE_VEHICLE', 'MAKE', 'INSR_TYPE']

df_encoded = df[categorical_cols].copy()
df_encoded['INSR_TYPE'] = df_encoded['INSR_TYPE'].apply(lambda x: str(x))

# Create one-hot encoded matrix
ps = '_!_'
df_binary = pd.get_dummies(df_encoded, prefix_sep=ps)

frequent_itemsets = apriori(df_binary, min_support=0.01, use_colnames=True)
print(frequent_itemsets)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5, return_metrics=['confidence'])


rules['antecedents'] = rules['antecedents'].apply(lambda x: parse_rule_items(x, ps))
rules['consequents'] = rules['consequents'].apply(lambda x: parse_rule_items(x, ps))

print(f"Found {len(rules)} rules")

      support                                           itemsets
0    0.164789        frozenset({USAGE_!_Fare Paying Passengers})
1    0.072433               frozenset({USAGE_!_General Cartage})
2    0.016484                        frozenset({USAGE_!_Others})
3    0.192912                     frozenset({USAGE_!_Own Goods})
4    0.045193                   frozenset({USAGE_!_Own service})
..        ...                                                ...
200  0.015334  frozenset({USAGE_!_Private, MAKE_!_YAMAHA, INS...
201  0.034140  frozenset({USAGE_!_Private, TYPE_VEHICLE_!_Tra...
202  0.032383  frozenset({MAKE_!_LADA, INSR_TYPE_!_1202, USAG...
203  0.052530  frozenset({MAKE_!_TOYOTA, TYPE_VEHICLE_!_Bus, ...
204  0.069227  frozenset({INSR_TYPE_!_1202, TYPE_VEHICLE_!_Mo...

[205 rows x 2 columns]
Found 355 rules


In [5]:
rules[rules['confidence'] == 1.0]

,antecedents,consequents,confidence
26,{'USAGE': 'Special Construction'},{'INSR_TYPE': 1202},1.0
39,{'MAKE': 'TVS'},{'TYPE_VEHICLE': 'Motor-cycle'},1.0
40,{'MAKE': 'YAMAHA'},{'TYPE_VEHICLE': 'Motor-cycle'},1.0
43,{'MAKE': 'CATERPILLAR'},{'TYPE_VEHICLE': 'Special construction'},1.0
53,{'MAKE': 'CATERPILLAR'},{'INSR_TYPE': 1202},1.0
64,"{'USAGE': 'Fare Paying Passengers', 'MAKE': 'D...",{'TYPE_VEHICLE': 'Bus'},1.0
102,"{'USAGE': 'Others', 'MAKE': 'BAJAJ'}",{'TYPE_VEHICLE': 'Motor-cycle'},1.0
151,"{'USAGE': 'Private', 'MAKE': 'BAJAJ'}",{'TYPE_VEHICLE': 'Motor-cycle'},1.0
152,"{'USAGE': 'Private', 'MAKE': 'YAMAHA'}",{'TYPE_VEHICLE': 'Motor-cycle'},1.0
172,"{'TYPE_VEHICLE': 'Special construction', 'USAG...",{'INSR_TYPE': 1202},1.0


In [6]:
rules.iloc[320]['antecedents']

{'MAKE': 'VOLKSWAGEN', 'TYPE_VEHICLE': 'Automobile'}

Rule 1:
  IF MAKE_LADA, USAGE_Taxi, INSR_TYPE_1202
  THEN TYPE_VEHICLE_Automobile
  Confidence: 1.000000 | Lift: 4.333165

Rule 2:
  IF INSR_TYPE_1201, MAKE_VOLKSWAGEN, TYPE_VEHICLE_Automobile
  THEN USAGE_Private
  Confidence: 0.99916 | Lift: 3.554573

Rule 3:
  IF USAGE_Taxi, MAKE_BAJAJ
  THEN frozenset({INSR_TYPE_1202})
  Confidence: 0.999838 | Lift: 1.398611

Rule 4:
  IF USAGE_Special Construction
  THEN INSR_TYPE_1202
  Confidence: 1.0 | Lift: 1.398837

Rule 5:
  IF MAKE_TVS
  THEN TYPE_VEHICLE_Motor-cycle
  Confidence: 1.0 | Lift: 4.378108

Применение аналогично

Пример конфига

In [ ]:
rules_zero = [
    {
        'antecedents': {'MAKE': 'LADA', 'USAGE': 'Taxi', 'INSR_TYPE': 1202},
        'consequent': {'TYPE_VEHICLE': 'Automobile'},
        'confidence': 1.000000
    },
    {
        'antecedents': {'MAKE': 'VOLKSWAGEN', 'TYPE_VEHICLE': 'Automobile', 'INSR_TYPE': 1201},
        'consequent': {'USAGE': 'Private'},
        'confidence': 0.99916
    },
    {
        'antecedents': {'MAKE': 'BAJAJ', 'USAGE': 'Taxi'},
        'consequent': {'INSR_TYPE': 1202},
        'confidence': 0.999838
    },
    {
        'antecedents': {'USAGE': 'Special Construction'},
        'consequent': {'INSR_TYPE': 1202},
        'confidence': 1.000000
    },
    {
        'antecedents': {'MAKE': 'TVS'},
        'consequent': {'TYPE_VEHICLE': 'Motor-cycle'},
        'confidence': 1.000000
    }
]

rules_else = [
    {
        'antecedents': {'MAKE': 'YAMAHA'},
        'consequent': {'TYPE_VEHICLE': 'Motor-cycle'},
        'confidence': 1.000000
    },
    {
        'antecedents': {'MAKE': 'CALABRESE'},
        'consequent': {'TYPE_VEHICLE': 'Trailers and semitrailers'},
        'confidence': 1.0
    },
    {
        'antecedents': {'MAKE': 'BAJAJ'},
        'consequent': {'TYPE_VEHICLE': 'Motor-cycle'},
        'confidence': 0.9974
    },
    {
        'antecedents': {'USAGE': 'Fare Paying Passengers'},
        'consequent': {'INSR_TYPE': 1202},
        'confidence': 0.998631
    },
    {
        'antecedents': {'USAGE': 'General Cartage'},
        'consequent': {'INSR_TYPE': 1202},
        'confidence': 0.9973
    }
]

apriori_config = {"INSR_ZERO": rules_zero, "ELSE": rules_else}

# import json
# with open('apriori_rules.json', 'w') as f:
#     json.dump(apriori_config, f)